In [ ]:
# Ensure gradalg is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# gradalg isn't already installed into this kernel.
try:
    import gradalg  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "gradalg" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import gradalg  # noqa: F401

# 05 — Cartan Calculus

Bu notebook [05_cartan_calculus.md](05_cartan_calculus.md) markdown'ının çalıştırılabilir sürümüdür. Beş Cartan bağıntısının `OperatorEquation` olarak inşası, `d² = 0` için axiom/theorem mod farkı, magic formülünün iki modda canlı ispatı, ve invariant-d helper'ı.

## Bundle

`CartanCalculus(d, L, ι, [·,·])` — dört ingredient tek objede.

In [ ]:
from gradalg.algebra.derivation import Derivation
from gradalg.brackets.lie import LieBracket
from gradalg.calculus.cartan import CartanCalculus, RELATIONS
from gradalg.calculus.exterior_algebra import ExteriorAlgebra
from gradalg.calculus.exterior_d import d
from gradalg.calculus.interior import interior
from gradalg.calculus.lie_derivative import lie_derivative
from gradalg.core.expr import Symbol
from gradalg.core.properties import Graded
from gradalg.core.registry import PropertyRegistry

cart = CartanCalculus(
    d=d, lie_derivative=lie_derivative,
    interior=interior, vector_bracket=LieBracket(),
)
print('RELATIONS:', RELATIONS)

## Beş bağıntı, beş `OperatorEquation`

In [ ]:
reg = PropertyRegistry()
f = Symbol("f")
reg.declare(f, Graded(degree=0))
algebra = ExteriorAlgebra((f,))
X = Derivation("X", degree=0)
Y = Derivation("Y", degree=0)

for name, kw in [
    ("d_squared_zero", {}),
    ("cartan_magic", {"X": X}),
    ("d_lie", {"X": X}),
    ("lie_lie", {"X": X, "Y": Y}),
    ("lie_iota", {"X": X, "Y": Y}),
]:
    eq = cart.relation(name, algebra=algebra, **kw)
    print(f"{name:16s}: {eq.lhs} = {eq.rhs}")

## `d² = 0` — axiom mode vs theorem mode

Sade helper `apply_d_squared_zero` her zaman 0'a çevirir. Default engine `d_squared_mode="theorem"` + foundational modda d(d(x)) → 0 rewrite'ını ProofStep olarak kaydeder.

In [ ]:
from gradalg.calculus.exterior_d import apply_d_squared_zero
from gradalg.proof.expansion import default_engine

x = Symbol("x")
reg.declare(x, Graded(degree=0))
print('axiom rewrite:', apply_d_squared_zero(d(d(x))))

engine = default_engine(
    registry=reg, mode="foundational", d_squared_mode="theorem"
)
expanded, steps = engine.expand(d(d(x)))
print('theorem-mode expanded:', expanded)
print('theorem-mode step rule:', steps[0].rule)

## Tüm beş bağıntı — `verify` üzerinden canlı ispat

`cartan_magic` iki modda da `ExteriorAlgebra((f,))` üstünde tek adımda kapanıyor; diğer dördü `AgreementOnGenerators` + `ExpandAndSimplify` zinciriyle generator seviyesinde kapanıyor.

In [ ]:
chain = cart.verify("cartan_magic", algebra=algebra, X=X, registry=reg)
print('efficient len:', len(chain), 'rule:', chain.steps[0].rule)

chain_f = cart.verify(
    "cartan_magic", algebra=algebra, X=X, registry=reg,
    mode="foundational",
)
print('foundational len:', len(chain_f), 'rule:', chain_f.steps[0].rule)

results = cart.verify_all(algebra=algebra, X=X, Y=Y, registry=reg)
for name, c in results.items():
    print(f'{name:16s}: len={len(c)}')

## `invariant_d` — magic + lie_iota türevi teorem

`dω(X, Y) = X(ω(Y)) − Y(ω(X)) − ω([X, Y])` — 1-formlar için Koszul-Cartan invariant formülü. `InvariantDOneFormDefinition`'ın default classification'ı `"theorem"` (d²=0'ın tersine) — çünkü formül doğal olarak magic + lie_iota'dan türüyor.

In [ ]:
from gradalg.calculus.invariant_d import invariant_d_one_form
from gradalg.brackets.lie import lie

omega = Symbol("ω")
reg.declare(omega, Graded(degree=1))
print(invariant_d_one_form(omega, X, Y, bracket=lie))

## Twisted Cartan bundle — `d_H = d + H∧`

Kapalı bir 3-form `H` ile H-twisted exterior derivative `d_H` Cartan calculus'ünü aynı beş bağıntıyla taşır. `TwistedCartanBundle(H)` wrapper'ı, `d_H`'yi taze bir `ExteriorDerivative` olarak inşa edip Lie-türevi factory'sini `d_H`'yi bundle slot'una geçecek şekilde kurar. Bundle inşa etmek `dH = 0` varsayımını yapmaktır.

In [ ]:
from gradalg.library import TwistedCartanBundle

H = Symbol("H")
reg.declare(H, Graded(degree=3))
bundle = TwistedCartanBundle(H)
print('bundle.d:', bundle.d)
print('bundle.cartan.d:', bundle.cartan.d)

algebra_H = ExteriorAlgebra((f,), d=bundle.d)
results = bundle.cartan.verify_all(
    algebra=algebra_H, X=X, Y=Y, registry=reg
)
for name, c in results.items():
    print(f'{name:16s}: len={len(c)}')

## Sonraki adım

Kendi bracket'iniz + Jacobi testi — [06_custom_bracket.md](06_custom_bracket.md) (Stage C).